In [ ]:
%pip install scikit-learn

In [ ]:

# from sklearn.linear_model import LogisticRegression

# 1. Clean your model data to drop missing values
model_data = sepsis_readm_severity.dropna(subset=[
    'readmitted_30d', 'drg_severity', 'has_mech_vent', 
    'has_high_cost_procedure'
])

# 2. Create dummy variables for categorical features (like drg_severity)
X = pd.get_dummies(
    model_data[['drg_severity', 'has_mech_vent', 'has_high_cost_procedure']], 
    drop_first=True
)
y = model_data['readmitted_30d']

# 3. Fit Logistic Regression with L2 Penalty (Ridge regularization) using scikit-learn
# penalty='l2' stops coefficients from blowing up to infinity due to separation
# C is the inverse of regularization strength (smaller C = stronger penalty)
lr_model = LogisticRegression(penalty='l2', C=1.0, solver='lbfgs')
lr_model.fit(X, y)

# 4. Calculate Standard Errors for Confidence Intervals (using the Hessian approximation)
# This calculates the inverse of the second derivative (Fisher Information Matrix)
X_design = np.hstack([np.ones((X.shape[0], 1)), X.values])
probabilities = lr_model.predict_proba(X_design[:, 1:])[:, 1]
# Weight matrix for logistic regression
W = probabilities * (1 - probabilities)
# Hessian matrix calculation
D = np.dot(X_design.T, X_design * W[:, np.newaxis])
# Add a small ridge penalty diagonal term matching C to prevent singular matrix errors
D += (1.0 / 1.0) * np.eye(D.shape[0]) 
cov_matrix = np.linalg.inv(D)
standard_errors = np.sqrt(np.diagonal(cov_matrix))[1:] # Exclude intercept

# 5. Compute 95% Confidence Intervals for Coefficients and Odds Ratios
coefs = lr_model.coef_[0]
z_score = 1.96  # 95% confidence level

ci_lower_coef = coefs - (z_score * standard_errors)
ci_upper_coef = coefs + (z_score * standard_errors)

# Exponentiate to get Odds Ratios and their respective Confidence Intervals
odds_ratios = np.exp(coefs)
ci_lower_or = np.exp(ci_lower_coef)
ci_upper_or = np.exp(ci_upper_coef)

# 6. Assemble into a clean results DataFrame
results_df = pd.DataFrame({
    'Variable': X.columns,
    'Odds_Ratio': odds_ratios,
    'CI_Lower': ci_lower_or,
    'CI_Upper': ci_upper_or,
    'Coefficient': coefs,
    'Std_Error': standard_errors
})

print(results_df)

results_df.to_csv('/Users/xumuyao/Desktop/JOBHUNTING/mockdata/hosp/sepsis_logistic_regression_results.csv', index=True)